##This version added closest found pixel latitude and longitude into the df

In [ ]:
#Mounting a new folder from google colab onto drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:

output_dir = "/content/drive/MyDrive/Pyrocb_data"
print(f"Output directory updated to: {output_dir}")

In [ ]:
import subprocess, sys

REQUIRED = ["goes2go", "s3fs", "xarray", "netCDF4", "h5netcdf", "pyproj", "numpy", "scipy"]

def install_if_missing(packages):
    for pkg in packages:
        try:
            __import__(pkg.replace("-", "_").split("[")[0])
        except ImportError:
            print(f"  Installing {pkg}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("Checking dependencies...")
install_if_missing(REQUIRED)
print("All dependencies ready.\n")


import os
import s3fs
import xarray as xr
import numpy as np
from datetime import datetime, timedelta
from pyproj import Proj


CONFIG = {

    "fire_name": "Sparks_Lake_BC",


    "date": datetime(2021, 8, 1),
    "start_hour_utc": 20,
    "end_hour_utc": 21,


    "fire_lat": 50.4,
    "fire_lon": -119.8,

    "satellite": 16,


    "output_dir": "/content/drive/MyDrive/Pyrocb_data",


    "crop_half_km": 100,
}


PYROCAST_BANDS = {
    "M6C01": "0.47µm  Blue",
    "M6C02": "0.64µm  Red",
    "M6C03": "0.86µm  Vegetation/Near-IR",
    "M6C07": "3.9µm   Shortwave IR (fire detection)",
    "M6C14": "11.2µm  Longwave IR (cloud-top temp)",
    "M6C16": "13.3µm  CO2 longwave IR",
}


def list_goes_files(fs, satellite: int, date: datetime, hour: int, band: str) -> list:
    """
    List all ABI-L1b Full Disk files on AWS for a given satellite, date, hour, and band.
    S3 path: noaa-goes{sat}/ABI-L1b-RadF/{year}/{doy}/{hour}/
    Files are filtered to the requested channel (e.g. C07).
    """
    bucket = f"noaa-goes{satellite}"
    doy = date.timetuple().tm_yday          # Day of year (1-365)
    path = f"{bucket}/ABI-L1b-RadF/{date.year}/{doy:03d}/{hour:02d}/"
    print(f"DEBUG: Attempting to list files from path: {path} for band: {band}") # Modified debug print

    try:
        all_files = fs.ls(path)
        print(f"DEBUG: Files found at {path} (before band filter): {all_files}") # Debug print
    except FileNotFoundError:
        print(f"    ⚠  No files found at: {path}")
        return []

    # Keep only files matching this band (e.g. "_C07_")
    matched = [f for f in all_files if f"-{band}_" in f]
    print(f"DEBUG: Matched files for band {band}: {matched}") # Added debug print
    return matched



def goes_projection(satellite: int):
    """
    Return a pyproj.Proj for the GOES ABI geostationary projection.
    GOES-16 sits at 75.2°W, GOES-17 at 137.2°W.
    """
    lon0 = -75.2 if satellite == 16 else -137.2
    return Proj(proj="geos", h=35786023.0, lon_0=lon0, sweep="x", ellps="GRS80")


def latlon_to_scan_radians(lat, lon, satellite):
    """
    Convert geographic lat/lon to GOES scan angle radians (x, y).
    """
    p = goes_projection(satellite)
    x_m, y_m = p(lon, lat)            # metres in geostationary projection
    H = 35786023.0 + 6378137.0        # satellite height + Earth radius
    x_rad = x_m / H                   # radians
    y_rad = y_m / H
    return x_rad, y_rad

def scan_radians_to_latlon(x_rad, y_rad, satellite):
    """
    Convert GOES scan angle radians (x, y) to geographic lat/lon.
    """
    p = goes_projection(satellite)
    H = 35786023.0 + 6378137.0
    x_m = x_rad * H
    y_m = y_rad * H
    lon, lat = p(x_m, y_m, inverse=True)
    return lat, lon


def download_and_crop(fs, s3_path: str, fire_lat: float, fire_lon: float,
                      satellite: int, crop_half_km: float, out_dir: str) -> str | None:
    """
    Stream a GOES NetCDF file from S3, crop to a box centred on the fire,
    and save to disk. Returns the output path or None on failure.
    """
    filename = os.path.basename(s3_path)
    out_path = os.path.join(out_dir, filename)

    if os.path.exists(out_path):
        print(f"    ✓  Already downloaded: {filename}")
        return out_path

    try:
        with fs.open(s3_path, "rb") as f:
            ds = xr.open_dataset(f, engine="h5netcdf")

            # ── Crop around fire location ──────────────────────────────
            # Convert fire lat/lon to scan angle radians
            x_fire, y_fire = latlon_to_scan_radians(fire_lat, fire_lon, satellite)

            # 100 km in scan radians (approx: 1 km ≈ 2.8e-5 rad at nadir)
            delta_rad = (crop_half_km * 1000) / (35786023.0 + 6378137.0)

            # GOES x coordinate increases eastward, y increases northward
            ds_crop = ds.sel(
                x=slice(x_fire - delta_rad, x_fire + delta_rad),
                y=slice(y_fire + delta_rad, y_fire - delta_rad)  # y is inverted
            )

            if ds_crop.dims.get("x", 0) == 0 or ds_crop.dims.get("y", 0) == 0:
                print(f"    ⚠  Fire location outside satellite view for {filename}")
                return None

            # Save cropped file
            ds_crop.to_netcdf(out_path)
            print(f"    ✓  Saved: {filename}  "
                  f"[{ds_crop.dims.get('x')}×{ds_crop.dims.get('y')} px]")
            return out_path

    except Exception as e:
        print(f"    ✗  Failed {filename}: {e}")
        return None



def load_multichannel_scene(file_paths_by_band: dict) -> np.ndarray | None:
    """
    Load 6 band NetCDF files for one timestamp and stack into
    a (6, H, W) numpy array — matching Pyrocast's (N, 200, 200) format.
    """
    arrays = []
    for band, path in sorted(file_paths_by_band.items()):
        ds = xr.open_dataset(path)
        # GOES L1b stores radiance in variable 'Rad'
        rad = ds["Rad"].values.astype(np.float32)
        arrays.append(rad)
        ds.close()

    if not arrays:
        return None


    min_h = min(a.shape[0] for a in arrays)
    min_w = min(a.shape[1] for a in arrays)
    arrays = [a[:min_h, :min_w] for a in arrays]

    scene = np.stack(arrays, axis=0)   # shape: (6, H, W)
    return scene


def main():
    # Use the global CONFIG dictionary
    global CONFIG

    print("=" * 60)
    print("  Pyrocast USA Satellite Imagery Downloader")
    print("=" * 60)
    print(f"  Fire       : {CONFIG['fire_name']}")
    print(f"  Date       : {CONFIG['date'].strftime('%Y-%m-%d')}")
    print(f"  Hours (UTC): {CONFIG['start_hour_utc']:02d}:00 – {CONFIG['end_hour_utc']:02d}:00")
    print(f"  Location   : {CONFIG['fire_lat']}°N, {CONFIG['fire_lon']}°W")
    print(f"  Satellite  : GOES-{CONFIG['satellite']}")
    print(f"  Crop box   : {CONFIG['crop_half_km']*2}×{CONFIG['crop_half_km']*2} km")
    print("=" * 60)

    # Create output directory

    fire_dir = os.path.join(CONFIG["output_dir"], CONFIG["fire_name"])
    os.makedirs(fire_dir, exist_ok=True)
    print(f"\nOutput folder: {fire_dir}\n")

    # Connect to AWS S3 (anonymous — no account needed)
    print("Connecting to AWS S3 (no account needed)...")
    fs = s3fs.S3FileSystem(anon=True)
    print("Connected.\n")

    # Summary stats
    total_files = 0
    failed_files = 0
    scenes = {}   # { hour: { band: filepath } }

    hours = range(CONFIG["start_hour_utc"], CONFIG["end_hour_utc"] + 1)

    for hour in hours:
        print(f"── Hour {hour:02d}:00 UTC ──────────────────────────────")
        hour_dir = os.path.join(fire_dir, f"hour_{hour:02d}UTC")
        os.makedirs(hour_dir, exist_ok=True)
        scenes[hour] = {}

        for band, desc in PYROCAST_BANDS.items():
            print(f"  Band {band} ({desc})")

            # List matching files on S3
            s3_files = list_goes_files(fs, CONFIG["satellite"], CONFIG["date"], hour, band)

            if not s3_files:
                print(f"    ⚠  No files found for {band} at hour {hour:02d}.")
                failed_files += 1
                continue

            # Take the first scan of the hour (closest to the hour mark)
            s3_path = s3_files[0]

            out_path = download_and_crop(
                fs, s3_path,
                fire_lat=CONFIG["fire_lat"],
                fire_lon=CONFIG["fire_lon"],
                satellite=CONFIG["satellite"],
                crop_half_km=CONFIG["crop_half_km"],
                out_dir=hour_dir
            )

            if out_path:
                scenes[hour][band] = out_path
                total_files += 1
            else:
                failed_files += 1

        print()

    # ── Stack into multi-channel arrays and save ───────────────────────
    print("Stacking bands into multi-channel arrays (6, H, W)...")
    for hour, band_files in scenes.items():
        if len(band_files) == len(PYROCAST_BANDS):
            array = load_multichannel_scene(band_files)
            if array is not None:
                npy_path = os.path.join(
                    fire_dir, f"scene_{CONFIG['date'].strftime('%Y%m%d')}_hour{hour:02d}UTC.npy"
                )
                np.save(npy_path, array)
                print(f"  ✓  Hour {hour:02d}: array shape {array.shape} → {os.path.basename(npy_path)}")
        else:
            print(f"  ⚠  Hour {hour:02d}: only {len(band_files)}/{len(PYROCAST_BANDS)} bands downloaded, skipping stack.")

    # ── Summary ───────────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("  DOWNLOAD COMPLETE")
    print("=" * 60)
    print(f"  Files downloaded : {total_files}")
    print(f"  Files failed     : {failed_files}")
    print(f"  Saved to         : {os.path.abspath(fire_dir)}")
    print()
    print("  Output files:")
    print("    ├── hour_20UTC/   ← per-band cropped NetCDF files")
    print("    ├── hour_21UTC/")
    print("    ├── ...")
    print(f"    └── scene_{CONFIG['date'].strftime('%Y%m%d')}_hour20UTC.npy  ← (6, H, W) array, ready for Pyrocast model")
    print()
    print("  Next step: Feed .npy arrays into the Pyrocast CNN model.")
    print("=" * 60)

#uncomment to download
# if __name__ == "__main__":
#     main()

Now we'll use the `cdsapi` client to download ERA5 data. We will retrieve 2m temperature, total precipitation, and surface pressure, which are commonly used meteorological variables. The download will cover the same date, time range, and a bounding box around the fire location as defined for the GOES data.

Once downloaded, you can load and inspect the ERA5 NetCDF file using `xarray`:

In [ ]:
from glob import glob
import os
import numpy as np
import xarray as xr


def process_pyrocb_hour_folder(hour_folder_path):
    """Navigates your exact 'Hour' folder structure, finds the 6 bands,

    and returns arrays perfectly formatted for Machine Learning.
    """
    # 1. Search for band files inside the specific hour folder
    # Uses wildcards (*C01*, *C02*, etc.) so you don't have to type the giant GOES names
    b01_file = glob(os.path.join(hour_folder_path, "*C01*.nc"))
    b02_file = glob(os.path.join(hour_folder_path, "*C02*.nc"))
    b03_file = glob(os.path.join(hour_folder_path, "*C03*.nc"))
    b07_file = glob(os.path.join(hour_folder_path, "*C07*.nc"))
    b14_file = glob(os.path.join(hour_folder_path, "*C14*.nc"))
    b16_file = glob(os.path.join(hour_folder_path, "*C16*.nc"))

    # Error checking to make sure you put all 6 files in the folder
    files = [b01_file, b02_file, b03_file, b07_file, b14_file, b16_file]
    if any(len(f) == 0 for f in files):
        raise FileNotFoundError(
            f"Missing one or more of the 6 required bands in: {hour_folder_path}"
        )



    # 2. Open the NetCDF datasets (grabbing index [0] from the glob list)
    ds01 = xr.open_dataset(b01_file[0])
    ds02 = xr.open_dataset(b02_file[0])
    ds03 = xr.open_dataset(b03_file[0])
    ds07 = xr.open_dataset(b07_file[0])
    ds14 = xr.open_dataset(b14_file[0])
    ds16 = xr.open_dataset(b16_file[0])

    # 3. Detect variable name ('Rad' for Level 1b, 'CMI' for Level 2)
    var_vis = "Rad" if "Rad" in ds01.variables else "CMI"
    var_ir = "Rad" if "Rad" in ds07.variables else "CMI"

    # Extract bands
    b01 = ds01[var_vis].astype(np.float32)
    b02 = ds02[var_vis].astype(np.float32)
    b03 = ds03[var_vis].astype(np.float32)
    t07 = ds07[var_ir].astype(np.float32)
    t14 = ds14[var_ir].astype(np.float32)
    t16 = ds16[var_ir].astype(np.float32)

    # 4. Standardize dimensions to match the 2km Thermal grid (t14)
    b01 = b01.interp_like(t14, method="nearest")
    b02 = b02.interp_like(t14, method="nearest")
    b03 = b03.interp_like(t14, method="nearest")

    # 5. Feature Engineering Math
    features = {
        "fire_proxy": t07 - t14,
        "cloud_height_proxy": t14 - t16,
        "simulated_green": (0.45 * b01) + (0.10 * b02) + (0.45 * b03),
        "raw_fire_bt": t07,
        "raw_cloud_bt": t14,
    }

    # NDVI math with safe zero-division handling
    denom = b03 + b02
    features["ndvi"] = np.where(denom == 0, 0, (b03 - b02) / denom)

    # 6. Format data channels
    feature_keys = list(features.keys())
    stacked_grid = np.stack([features[k] for k in feature_keys], axis=-1)

    # Flatten grid for tabular ML models (Pixels, Channels)
    tabular_matrix = stacked_grid.reshape(-1, len(feature_keys))

    # Close file streams safely
    for ds in [ds01, ds02, ds03, ds07, ds14, ds16]:
        ds.close()

    return stacked_grid, tabular_matrix, feature_keys

target_folder = "/content/drive/MyDrive/Pyrocb_data/Sparks_Lake_fire_BC_2021/hour_20UTC"
grid, ml_input, features_list = process_pyrocb_hour_folder(target_folder)
print(grid)

In [ ]:
from datetime import datetime, timedelta
import re
from pyproj import Proj # Import Proj for goes_projection

#/content/drive/MyDrive/Pyrocb_data/Sparks_Lake_fire_BC_2021/hour_20UTC/OR_ABI-L1b-RadF-M6C01_G17_s20212132000321_e20212132009388_c20212132009415.nc

def goes_projection(satellite: int):
    """
    Return a pyproj.Proj for the GOES ABI geostationary projection.
    GOES-16 sits at 75.2°W, GOES-17 at 137.2°W.
    """
    lon0 = -75.2 if satellite == 16 else -137.2
    return Proj(proj="geos", h=35786023.0, lon_0=lon0, sweep="x", ellps="GRS80")

def scan_radians_to_latlon(x_rad, y_rad, satellite):
    """
    Convert GOES scan angle radians (x, y) to geographic lat/lon.
    """
    p = goes_projection(satellite)
    H = 35786023.0 + 6378137.0
    x_m = x_rad * H
    y_m = y_rad * H
    lon, lat = p(x_m, y_m, inverse=True)
    return lat, lon


def process_pyrocb_hour_folder_with_timestamp(hour_folder_path, satellite_id: int):
    """
    Navigates your exact 'Hour' folder structure, finds the 6 bands,
    and returns arrays perfectly formatted for Machine Learning, along with the timestamp
    and pixel-level latitude and longitude.
    """
    # 1. Search for band files inside the specific hour folder
    b01_file = glob(os.path.join(hour_folder_path, "*C01*.nc"))
    b02_file = glob(os.path.join(hour_folder_path, "*C02*.nc"))
    b03_file = glob(os.path.join(hour_folder_path, "*C03*.nc"))
    b07_file = glob(os.path.join(hour_folder_path, "*C07*.nc"))
    b14_file = glob(os.path.join(hour_folder_path, "*C14*.nc"))
    b16_file = glob(os.path.join(hour_folder_path, "*C16*.nc"))

    # Error checking
    files = [b01_file, b02_file, b03_file, b07_file, b14_file, b16_file]
    if any(len(file) == 0 for file in files):
        # We'll just return None for data and features, but keep the timestamp as None
        print(f"  ⚠ Missing one or more of the 6 required bands in: {hour_folder_path}. Skipping.")
        return None, None, None, None, None, None

    # Extract timestamp from one of the filenames (e.g., b01_file[0])
    filename_b01 = os.path.basename(b01_file[0])
    match = re.search(r"_s(\d{13})", filename_b01)
    timestamp = None
    if match:
        timestamp_str = match.group(1) # YYYYJJJHHMMSS
        year = int(timestamp_str[0:4])
        day_of_year = int(timestamp_str[4:7])
        hour = int(timestamp_str[7:9])
        minute = int(timestamp_str[9:11])
        second = int(timestamp_str[11:13])
        timestamp = datetime(year, 1, 1) + timedelta(days=day_of_year - 1, hours=hour, minutes=minute, seconds=second)

    # 2. Open the NetCDF datasets
    ds01 = xr.open_dataset(b01_file[0])
    ds02 = xr.open_dataset(b02_file[0])
    ds03 = xr.open_dataset(b03_file[0])
    ds07 = xr.open_dataset(b07_file[0])
    ds14 = xr.open_dataset(b14_file[0])
    ds16 = xr.open_dataset(b16_file[0])

    # 3. Detect variable name ('Rad' for Level 1b, 'CMI' for Level 2)
    var_vis = "Rad" if "Rad" in ds01.variables else "CMI"
    var_ir = "Rad" if "Rad" in ds07.variables else "CMI"

    # Extract bands
    b01 = ds01[var_vis].astype(np.float32)
    b02 = ds02[var_vis].astype(np.float32)
    b03 = ds03[var_vis].astype(np.float32)
    t07 = ds07[var_ir].astype(np.float32)
    t14 = ds14[var_ir].astype(np.float32)
    t16 = ds16[var_ir].astype(np.float32)

    # 4. Standardize dimensions to match the 2km Thermal grid (t14)
    # This also ensures all bands have the same 'x' and 'y' coordinates
    b01 = b01.interp_like(t14, method="nearest")
    b02 = b02.interp_like(t14, method="nearest")
    b03 = b03.interp_like(t14, method="nearest")

    # Get scan angle coordinates (x, y) from the interpolated grid (e.g., t14)
    x_scan_rad = t14.x.values
    y_scan_rad = t14.y.values

    # Create 2D grids for x and y scan angles
    X_rad, Y_rad = np.meshgrid(x_scan_rad, y_scan_rad)

    # Convert scan angles to lat/lon
    lat_grid, lon_grid = scan_radians_to_latlon(X_rad, Y_rad, satellite_id)

    # 5. Feature Engineering Math
    features = {
        "fire_proxy": t07 - t14,
        "cloud_height_proxy": t14 - t16,
        "simulated_green": (0.45 * b01) + (0.10 * b02) + (0.45 * b03),
        "raw_fire_bt": t07,
        "raw_cloud_bt": t14,
    }

    # NDVI math with safe zero-division handling
    denom = b03 + b02
    features["ndvi"] = np.where(denom == 0, 0, (b03 - b02) / denom)

    # 6. Format data channels
    feature_keys = list(features.keys())
    stacked_grid = np.stack([features[k] for k in feature_keys], axis=-1)

    # Flatten grid for tabular ML models (Pixels, Channels)
    tabular_matrix = stacked_grid.reshape(-1, len(feature_keys))

    # Flatten lat/lon grids to match the tabular_matrix
    lat_pixels = lat_grid.flatten()
    lon_pixels = lon_grid.flatten()

    # Close file streams safely
    for ds in [ds01, ds02, ds03, ds07, ds14, ds16]:
        ds.close()

    return stacked_grid, tabular_matrix, feature_keys, timestamp, lat_pixels, lon_pixels

In [ ]:
import pandas as pd
import os
from glob import glob
import xarray as xr
import numpy as np

def load_and_process_all_fire_data(base_output_dir: str) -> pd.DataFrame:
    """
    Loads and processes all hourly data from all fire events found in the base_output_dir
    into a single Pandas DataFrame suitable for ML training.
    """
    all_data_frames = []

    # Iterate through each fire event folder (e.g., 'Sparks_Lake_fire_BC_2021')
    for fire_event_folder in os.listdir(base_output_dir):
        fire_event_path = os.path.join(base_output_dir, fire_event_folder)
        if os.path.isdir(fire_event_path):
            # Exclude .ipynb_checkpoints from processing
            if fire_event_folder == ".ipynb_checkpoints":
                print(f"Skipping .ipynb_checkpoints directory: {fire_event_folder}")
                continue

            print(f"Processing fire event: {fire_event_folder}")

            # For a given fire event, we need the satellite_id to convert scan radians to lat/lon.
            # This info is not directly in the folder name, but typically part of the config
            # or derived from the filename. For simplicity, we'll use a placeholder or assume G17 from filename.
            # A more robust solution would be to pass the config or extract from filename during download.
            # Here, we'll try to infer from the first file found.
            satellite_id = 17 # Defaulting to GOES-17 as seen in example filenames

            # Iterate through each hour folder within the fire event (e.g., 'hour_20UTC')
            for hour_folder in os.listdir(fire_event_path):
                if hour_folder.startswith('hour_') and os.path.isdir(os.path.join(fire_event_path, hour_folder)):
                    hour_folder_path = os.path.join(fire_event_path, hour_folder)
                    print(f"  Processing hour folder: {hour_folder_path}")

                    # Process the data for this hour, including lat/lon
                    stacked_grid, tabular_matrix, feature_keys, timestamp, lat_pixels, lon_pixels = \
                        process_pyrocb_hour_folder_with_timestamp(hour_folder_path, satellite_id)

                    if tabular_matrix is not None:
                        # Create a DataFrame for the current hour's data
                        df_hour = pd.DataFrame(tabular_matrix, columns=feature_keys)

                        # Add metadata columns: fire_name, timestamp, pixel lat/lon
                        df_hour['fire_name'] = fire_event_folder
                        df_hour['timestamp'] = timestamp
                        df_hour['hour_utc'] = timestamp.hour if timestamp else None
                        df_hour['pixel_latitude'] = lat_pixels
                        df_hour['pixel_longitude'] = lon_pixels

                        # print(df_hour['timestamp'].head()) # Print head to reduce verbosity, but still check

                        all_data_frames.append(df_hour)
                    else:
                        print(f"  Skipping {hour_folder_path} due to missing bands or timestamp extraction failure.")

    # Concatenate all hourly DataFrames into one large DataFrame
    if all_data_frames:
        combined_df = pd.concat(all_data_frames, ignore_index=True)
        return combined_df
    else:
        print("No data processed to combine.")
        return pd.DataFrame()

# --- Demonstration --- #
# Assuming output_dir is defined from previous cells
print(f"\nAggregating data from all fire events in: {CONFIG['output_dir']}")
combined_ml_df = load_and_process_all_fire_data(CONFIG['output_dir'])
shuffled_df = combined_ml_df.sample(frac=1, random_state=42).reset_index(drop=True)
print(f"this is first five rows of shuffled: {shuffled_df.head()}")
print("\n--- Combined DataFrame for Machine Learning ---")
print(f"Total rows (pixels across all hours and fires): {combined_ml_df.shape[0]}")
print(f"Total columns: {combined_ml_df.shape[1]}")
print("First 5 rows of the combined DataFrame:")
display(combined_ml_df.tail())

# Add a dedicated test call to surface debug prints
print("\n--- Debugging process_pyrocb_hour_folder_with_timestamp ---")
example_hour_path = os.path.join(CONFIG['output_dir'], "Sparks_Lake_fire_BC_2021", "hour_20UTC")
print(f"Testing with example path: {example_hour_path}")
_, _, _, debug_timestamp, debug_lat, debug_lon = process_pyrocb_hour_folder_with_timestamp(example_hour_path, satellite_id=17)
print(f"Resulting timestamp from debug call: {debug_timestamp}")
print(f"Sample pixel lat/lon from debug call: {debug_lat[0]}, {debug_lon[0]}")
print("--- End Debugging ---\n")

In [ ]:
from math import radians, sin, cos, sqrt, atan2

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371  # Radius of Earth in kilometers

    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

# Define the target latitude and longitude using the original CONFIG values
target_lat = CONFIG['fire_lat']
target_lon = CONFIG['fire_lon']

print(f"Finding closest point to Target Latitude: {target_lat}, Longitude: {target_lon}")

# Calculate distances for all points in the DataFrame
distances = combined_ml_df.apply(lambda row: haversine_distance(target_lat, target_lon, row['pixel_latitude'], row['pixel_longitude']), axis=1)

# Find the index of the minimum distance
closest_idx = distances.idxmin()

# Get the row with the closest point
closest_point_row = combined_ml_df.loc[closest_idx]
closest_distance = distances.loc[closest_idx]

print(f"\nClosest point found (from original DataFrame):")
print(f"  Latitude: {closest_point_row['pixel_latitude']}")
print(f"  Longitude: {closest_point_row['pixel_longitude']}")
print(f"  Distance: {closest_distance:.2f} km")

display(closest_point_row.to_frame().T)


## GENERATION AS PER PROMPT

In [ ]:
#Mounting a new folder from google colab onto drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import subprocess, sys

# List of required packages for GOES processing and feature extraction
REQUIRED = ["goes2go", "s3fs", "xarray", "netCDF4", "h5netcdf", "pyproj", "numpy", "scipy", "pandas", "openpyxl"]

def install_if_missing(packages):
    for pkg in packages:
        try:
            # Try to import the package to check if it exists
            __import__(pkg.replace("-", "_").split("[")[0])
        except ImportError:
            print(f"LOG: Installing missing dependency: {pkg}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("LOG: Checking system dependencies...")
install_if_missing(REQUIRED)
print("LOG: All dependencies ready.\n")

LOG: Checking system dependencies...
LOG: All dependencies ready.



In [ ]:
import pandas as pd
import numpy as np
import os
import s3fs
import xarray as xr
import re
from datetime import datetime, timedelta
from math import radians, sin, cos, sqrt, atan2
from pyproj import Proj
from glob import glob

# --- 0. Constants & Config ---
EXCEL_FILE = '/content/drive/MyDrive/Pyrocb_data/pyrocb_events_v0.xlsx'
BASE_DRIVE_PATH = '/content/drive/MyDrive/Pyrocb_data/GOES/'
ROWS_TO_PROCESS = 4

PYROCAST_BANDS = {
    "M6C01": "Blue", "M6C02": "Red", "M6C03": "Veg",
    "M6C07": "Fire", "M6C14": "Cloud", "M6C16": "CO2"
}

# --- 1. Helper Functions ---
def goes_projection(satellite):
    lon0 = -75.2 if satellite == 16 else -137.2
    return Proj(proj='geos', h=35786023.0, lon_0=lon0, sweep='x', ellps='GRS80')

def scan_radians_to_latlon(x_rad, y_rad, satellite):
    p = goes_projection(satellite)
    H = 35786023.0 + 6378137.0
    lon, lat = p(x_rad * H, y_rad * H, inverse=True)
    return lat, lon

def latlon_to_scan_radians(lat, lon, satellite):
    p = goes_projection(satellite)
    x_m, y_m = p(lon, lat)
    H = 35786023.0 + 6378137.0
    return x_m / H, y_m / H

def haversine_distance(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    d = sin((lat2-lat1)/2)**2 + cos(lat1)*cos(lat2)*sin((lon2-lon1)/2)**2
    return 6371 * 2 * atan2(sqrt(d), sqrt(1-d))

def download_and_crop(fs, s3_path, lat, lon, sat, half_km, out_dir):
    fname = os.path.basename(s3_path)
    dest = os.path.join(out_dir, fname)
    if os.path.exists(dest): return dest
    try:
        with fs.open(s3_path, 'rb') as f:
            ds = xr.open_dataset(f, engine='h5netcdf')
            x_f, y_f = latlon_to_scan_radians(lat, lon, sat)
            dr = (half_km * 1000) / (35786023.0 + 6378137.0)
            ds_c = ds.sel(x=slice(x_f-dr, x_f+dr), y=slice(y_f+dr, y_f-dr))
            ds_c.to_netcdf(dest)
            return dest
    except Exception: return None

def process_features(folder, sat):
    files = {b: glob(os.path.join(folder, f"*{b}*.nc")) for b in PYROCAST_BANDS}
    if any(len(f) == 0 for f in files.values()): return None

    datasets = {b: xr.open_dataset(files[b][0]) for b in files}
    ts = None
    match = re.search(r'_s(\d{13})', os.path.basename(files['M6C01'][0]))
    if match:
        s = match.group(1)
        ts = datetime(int(s[0:4]), 1, 1) + timedelta(days=int(s[4:7])-1, hours=int(s[7:9]), minutes=int(s[9:11]))

    b = {k: datasets[k]['Rad'].astype(np.float32).interp_like(datasets['M6C14'], method='nearest') for k in ['M6C01','M6C02','M6C03']}
    t07, t14, t16 = datasets['M6C07']['Rad'].values, datasets['M6C14']['Rad'].values, datasets['M6C16']['Rad'].values

    feat = {
        "fire_proxy": t07 - t14, "cloud_height_proxy": t14 - t16,
        "simulated_green": (0.45*b['M6C01']) + (0.10*b['M6C02']) + (0.45*b['M6C03']),
        "raw_fire_bt": t07, "raw_cloud_bt": t14
    }

    keys = list(feat.keys())
    grid = np.stack([feat[k] for k in keys], axis=-1)
    X, Y = np.meshgrid(datasets['M6C14'].x.values, datasets['M6C14'].y.values)
    lats, lons = scan_radians_to_latlon(X, Y, sat)

    for d in datasets.values(): d.close()
    return grid.reshape(-1, len(keys)), keys, ts, lats.flatten(), lons.flatten()

In [ ]:
print(f"LOG: Loading Excel: {EXCEL_FILE}")
events = pd.read_excel(EXCEL_FILE).head(ROWS_TO_PROCESS)
fs = s3fs.S3FileSystem(anon=True)
results = []

for _, row in events.iterrows():
    pid, lat_orig, lon_orig = str(row['pyroCb_id']), row['pyroCb_latitude'], row['pyroCb_longitude']
    start = pd.to_datetime(row['extract_ini_date_utc'])
    days = int(row['length_fire'])
    print(f"LOG: Event {pid} | Duration {days} days")

    for d_idx in range(1, days + 1):
        curr_date = start + timedelta(days=d_idx-1)
        for offset in [0, 6, 12, 18]:
            t_time = curr_date.replace(hour=start.hour) + timedelta(hours=offset)
            h_str = f"hour_{t_time.hour:02d}UTC"
            path = os.path.join(BASE_DRIVE_PATH, pid, f"Day_{d_idx}", h_str)
            os.makedirs(path, exist_ok=True)

            print(f"  LOG: Fetching {h_str} for {t_time.date()}")
            downloads = {}
            for b in PYROCAST_BANDS:
                s3_bucket = f"noaa-goes17/ABI-L1b-RadF/{t_time.year}/{t_time.timetuple().tm_yday:03d}/{t_time.hour:02d}/"
                try:
                    match = [f for f in fs.ls(s3_bucket) if f"-{b}_" in f]
                    if match: downloads[b] = download_and_crop(fs, match[0], lat_orig, lon_orig, 17, 100, path)
                except: continue

            if len(downloads) == 6:
                tab, keys, ts, lats, lons = process_features(path, 17)
                # Find closest pixel
                dists = [haversine_distance(lat_orig, lon_orig, lt, ln) for lt, ln in zip(lats, lons)]
                idx = np.argmin(dists)

                res = {
                    'pyroCb_id': pid,
                    'day': d_idx,
                    'timestamp': ts,
                    'pixel_latitude': lats[idx],
                    'pixel_longitude': lons[idx],
                    'dist_km': dists[idx]
                }
                for i, k in enumerate(keys): res[k] = tab[idx, i]
                results.append(res)
                print(f"    LOG: Extracted pixel at {dists[idx]:.2f}km")

final_df = pd.DataFrame(results)
final_df.to_csv('/content/drive/MyDrive/pyrocb_processed_results.csv', index=False)
display(final_df)

LOG: Loading Excel: /content/drive/MyDrive/Pyrocb_data/pyrocb_events_v0.xlsx
LOG: Event 253 | Duration 6 days
  LOG: Fetching hour_04UTC for 2022-03-29


/tmp/ipykernel_63907/3561265630.py:26: SerializationWarning: saving variable y with floating point data as an integer dtype without any _FillValue to use for NaNs
  if match: downloads[b] = download_and_crop(fs, match[0], lat_orig, lon_orig, 17, 100, path)
/tmp/ipykernel_63907/3561265630.py:26: SerializationWarning: saving variable x with floating point data as an integer dtype without any _FillValue to use for NaNs
  if match: downloads[b] = download_and_crop(fs, match[0], lat_orig, lon_orig, 17, 100, path)
/tmp/ipykernel_63907/3561265630.py:26: SerializationWarning: saving variable y with floating point data as an integer dtype without any _FillValue to use for NaNs
  if match: downloads[b] = download_and_crop(fs, match[0], lat_orig, lon_orig, 17, 100, path)
/tmp/ipykernel_63907/3561265630.py:26: SerializationWarning: saving variable x with floating point data as an integer dtype without any _FillValue to use for NaNs
  if match: downloads[b] = download_and_crop(fs, match[0], lat_ori

    LOG: Extracted pixel at 2.08km
  LOG: Fetching hour_10UTC for 2022-03-29


/tmp/ipykernel_63907/3561265630.py:26: SerializationWarning: saving variable y with floating point data as an integer dtype without any _FillValue to use for NaNs
  if match: downloads[b] = download_and_crop(fs, match[0], lat_orig, lon_orig, 17, 100, path)
/tmp/ipykernel_63907/3561265630.py:26: SerializationWarning: saving variable x with floating point data as an integer dtype without any _FillValue to use for NaNs
  if match: downloads[b] = download_and_crop(fs, match[0], lat_orig, lon_orig, 17, 100, path)
/tmp/ipykernel_63907/3561265630.py:26: SerializationWarning: saving variable y with floating point data as an integer dtype without any _FillValue to use for NaNs
  if match: downloads[b] = download_and_crop(fs, match[0], lat_orig, lon_orig, 17, 100, path)
/tmp/ipykernel_63907/3561265630.py:26: SerializationWarning: saving variable x with floating point data as an integer dtype without any _FillValue to use for NaNs
  if match: downloads[b] = download_and_crop(fs, match[0], lat_ori

    LOG: Extracted pixel at 2.08km
  LOG: Fetching hour_16UTC for 2022-03-29


...

/tmp/ipykernel_63907/3561265630.py:26: SerializationWarning: saving variable y with floating point data as an integer dtype without any _FillValue to use for NaNs
  if match: downloads[b] = download_and_crop(fs, match[0], lat_orig, lon_orig, 17, 100, path)
/tmp/ipykernel_63907/3561265630.py:26: SerializationWarning: saving variable x with floating point data as an integer dtype without any _FillValue to use for NaNs
  if match: downloads[b] = download_and_crop(fs, match[0], lat_orig, lon_orig, 17, 100, path)
/tmp/ipykernel_63907/3561265630.py:26: SerializationWarning: saving variable y with floating point data as an integer dtype without any _FillValue to use for NaNs
  if match: downloads[b] = download_and_crop(fs, match[0], lat_orig, lon_orig, 17, 100, path)
/tmp/ipykernel_63907/3561265630.py:26: SerializationWarning: saving variable x with floating point data as an integer dtype without any _FillValue to use for NaNs
  if match: downloads[b] = download_and_crop(fs, match[0], lat_ori

    LOG: Extracted pixel at 1.98km
  LOG: Fetching hour_23UTC for 2021-07-13


/tmp/ipykernel_63907/3561265630.py:26: SerializationWarning: saving variable y with floating point data as an integer dtype without any _FillValue to use for NaNs
  if match: downloads[b] = download_and_crop(fs, match[0], lat_orig, lon_orig, 17, 100, path)
/tmp/ipykernel_63907/3561265630.py:26: SerializationWarning: saving variable x with floating point data as an integer dtype without any _FillValue to use for NaNs
  if match: downloads[b] = download_and_crop(fs, match[0], lat_orig, lon_orig, 17, 100, path)
/tmp/ipykernel_63907/3561265630.py:26: SerializationWarning: saving variable y with floating point data as an integer dtype without any _FillValue to use for NaNs
  if match: downloads[b] = download_and_crop(fs, match[0], lat_orig, lon_orig, 17, 100, path)
/tmp/ipykernel_63907/3561265630.py:26: SerializationWarning: saving variable x with floating point data as an integer dtype without any _FillValue to use for NaNs
  if match: downloads[b] = download_and_crop(fs, match[0], lat_ori

    LOG: Extracted pixel at 1.98km


,pyroCb_id,day,timestamp,pixel_latitude,pixel_longitude,dist_km,fire_proxy,cloud_height_proxy,simulated_green,raw_fire_bt,raw_cloud_bt
0,253,1,2022-03-29 04:00:00,25.601092,-80.430891,2.075548,-105.325630,7.693130,0.082010,0.650714,105.976341
1,253,1,2022-03-29 10:00:00,25.601092,-80.430891,2.075548,-105.450645,8.545845,6.305539,0.674180,106.124825
2,253,1,2022-03-29 16:00:00,25.601092,-80.430891,2.075548,-102.601715,9.669479,141.742645,0.949506,103.551224
3,253,1,2022-03-29 22:00:00,25.601092,-80.430891,2.075548,-104.065018,6.323593,154.588150,1.069960,105.134979
4,253,2,2022-03-30 04:00:00,25.601092,-80.430891,2.075548,-108.993744,10.700821,0.050292,0.694516,109.688263
...,...,...,...,...,...,...,...,...,...,...,...
89,202,5,2021-07-12 23:00:00,50.797869,-95.027972,1.980107,-82.788055,7.049255,185.798920,1.065267,83.853325
90,202,6,2021-07-13 05:00:00,50.797869,-95.027972,1.980107,-108.767616,12.263428,-0.151037,0.722675,109.490288
91,202,6,2021-07-13 11:00:00,50.797869,-95.027972,1.980107,-99.723198,7.635406,0.050292,0.561546,100.284744
92,202,6,2021-07-13 17:00:00,50.797869,-95.027972,1.980107,-102.076805,17.042023,89.013771,1.276455,103.353256


#Small code asked to compare features and see patterns in data  ( only for first fire (260))

In [ ]:
import pandas as pd
import os
# Load the processed results to analyze the patterns manually
results_path = '/content/drive/MyDrive/pyrocb_processed_results.csv'
if os.path.exists(results_path):
    final_df = pd.read_csv(results_path)
    final_df['timestamp'] = pd.to_datetime(final_df['timestamp'])

    # Display the full dataframe for manual inspection
    print("LOG: Processed Results for Analysis:")
    display(final_df.sort_values('timestamp'))

    # Calculate some quick deltas to help pinpoint the 6/11 shift
    print("\n--- Analysis: Changes around 6/11 ---")
    pivot = final_df.copy()
    pivot['is_pyrocb_phase'] = pivot['timestamp'] >= '2022-06-11'
    summary = pivot.groupby('is_pyrocb_phase')[['fire_proxy', 'cloud_height_proxy', 'raw_cloud_bt']].mean()
    display(summary)
else:
    print("LOG: CSV file not found at /content/drive/MyDrive/pyrocb_processed_results.csv")